# Trimer Case2:  Shared codes

This page contains parameter values and functions used in other pages of the chapter.

In [39]:
import matplotlib.pyplot as plt
import numpy as np
from qutip import *

**Constant parameters**

In [40]:
# default energy scale
omega0 = 1.0

# default spontaneous emission rate
gamma0 = 0.1

def scale(omega0):
    k0 = omega0  # omega = c k, c=1
    lambda0 = 2*np.pi/k0
    return k0, lambda0

def planck_dist(omega0,T):
    return 1/(np.exp(omega0/T)-1) # Planck distributio

**Spin operators in 2^3 = 8 dimensions**

In [41]:
# Kronecker products of three operators
def kprod3(o1,o2,o3):
    return tensor(tensor(o1,o2),o3)

# Spin operators ---
def spin_ops():
    i2 = qeye(2)
    sz = [kprod3(sigmaz(),i2,i2),kprod3(i2,sigmaz(),i2),kprod3(i2,i2,sigmaz())]
    sp = [kprod3(sigmap(),i2,i2),kprod3(i2,sigmap(),i2),kprod3(i2,i2,sigmap())]
    sm = [kprod3(sigmam(),i2,i2),kprod3(i2,sigmam(),i2),kprod3(i2,i2,sigmam())]
    return sz, sp, sm

**Product basis**

In [42]:
def pbasis():
    #--This function calls kprod3--
    # v = product basis vectors [8][8]
    # label = names of corresponding states [8]
    label=["eee","eeg","ege","egg","gee","geg","gge","ggg"]
    v=[]
    v.append(kprod3(basis(2,0),basis(2,0),basis(2,0)))
    v.append(kprod3(basis(2,0),basis(2,0),basis(2,1)))
    v.append(kprod3(basis(2,0),basis(2,1),basis(2,0)))
    v.append(kprod3(basis(2,0),basis(2,1),basis(2,1)))
    v.append(kprod3(basis(2,1),basis(2,0),basis(2,0)))
    v.append(kprod3(basis(2,1),basis(2,0),basis(2,1)))
    v.append(kprod3(basis(2,1),basis(2,1),basis(2,0)))
    v.append(kprod3(basis(2,1),basis(2,1),basis(2,1)))
    return v, label

**Dicke basis**

In [43]:
def dicke(v):
    #--- v = product basis [8][8]
    #--- u = Dicke basis [8][8]
    u=[]
    u.append(v[0])
    u.append(1/np.sqrt(3)*(v[1]+v[2]+v[4]))
    u.append(1/np.sqrt(6)*(2*v[1]-v[2]-v[4]))
    u.append(1/np.sqrt(2)*(v[4]-v[2]))
    u.append(1/np.sqrt(3)*(v[3]+v[5]+v[6]))
    u.append(1/np.sqrt(6)*(2*v[6]-v[5]-v[3]))
    u.append(1/np.sqrt(2)*(v[3]-v[5]))
    u.append(v[7]) 
    return u

**Hamiltonian**

In [44]:
def hamiltonian(gamma0,a):
    # H = Hamiltonian [8,8]
    
    # free Hamiltonian
    H = omega0/2 * sum(sz) 
    
    # dipole-dipole coupling
    x = k0*a
    Omega = -3/4*gamma0*( np.cos(x)/x * (1-1/x**2) - np.sin(x)/x**2 )

    for i in range(3):
        for j in range(i+1,3):
            H += Omega*(sp[i]*sm[j]+sp[j]*sm[i])

    return H, Omega

**Decay rates**

In [52]:
def decay_rate(gamma0,a):
    #  Gamma = off diagonal element of the Gamma matrix.
    #  gamma = transition rate for each channel [3]
    x = k0*a
    Gamma = (3/2)*gamma0*( np.sin(x)/x * (1-1/x**2) + np.cos(x)/x**2 )
    gamma = [gamma0+2*Gamma, gamma0-Gamma, gamma0-Gamma]
    I0 = 3*gamma0/(8*np.pi)
    return Gamma, gamma, I0

**Jump and Collaps operators**

In [46]:
def collapse_ops(gamma,NT):
    # L = emission and absorption operators [6]
    # c_ops = collapse operators including the decay rates [6]
    w1 = np.exp(2j*np.pi/3)
    w2 = w1**2
    L1=1/np.sqrt(3)*(sm[0]+sm[1]+sm[2])
    L2=1/np.sqrt(3)*(sm[0]+w1*sm[1]+w2*sm[2])
    L3=1/np.sqrt(3)*(sm[0]+w2*sm[1]+w1*sm[2])
    L4=L1.dag()
    L5=L2.dag()
    L6=L3.dag()
    L = [L1,L2,L3,L4,L5,L6]
    c_ops = []
    c_ops.append(np.sqrt(gamma[0]*(NT+1))*L1)
    c_ops.append(np.sqrt(gamma[1]*(NT+1))*L2)
    c_ops.append(np.sqrt(gamma[2]*(NT+1))*L3)
    c_ops.append(np.sqrt(gamma[0]*NT)*L4)
    c_ops.append(np.sqrt(gamma[1]*NT)*L5)
    c_ops.append(np.sqrt(gamma[2]*NT)*L6)
    return L, c_ops

**Energy eigenvalues**

In [47]:
def eigen_energies(H):
    # W = numerical eigenvalues [8]
    # E = theoretical values [8]
    # order = index to sort E in descending order [8]
    E=[3*omega0/2, omega0/2+2*Omega, 0.5*omega0-Omega, 0.5*omega0-Omega, -0.5*omega0+2*Omega, -0.5*omega0-Omega,  -0.5*omega0-Omega, -3*omega0/2]
    W=H.eigenenergies()
    order = np.argsort(E, descending=True)
    W = np.sort(W)
    return np.array(W), np.array(E), order

**Transition energy**

In [48]:
def transition_energies(E):
    # w = transition energies for each channel [3]
    w0 = np.array([E[0]-E[1], E[1]-E[4], E[4]-E[7]])
    w1 = np.array([E[0]-E[2], E[2]-E[6], E[6]-E[7]])
    w = [w0,w1,w1]
    return w

**position of emitters**

In [51]:
def emitters_pos(a):
    # returns the position of emitters as numpy array [3][3]
    # coordinate origin = center of the triangle
    r1 = np.array([a / np.sqrt(3.0), 0.0, 0.0])
    r2 = np.array([-a / (2.0 * np.sqrt(3.0)), +a / 2.0, 0.0])
    r3 = np.array([-a / (2.0 * np.sqrt(3.0)), -a / 2.0, 0.0])
    return np.array([r1,r2,r3])


**location of detector**

In [50]:
def detector_pos(theta,phi):

    n=np.array([np.sin(theta)*np.cos(phi),np.sin(theta)*np.sin(phi),np.cos(theta)])

    return n